# Coherence and lag-sensitive connectivity
Run after installing `pip install -e ".[dev]"`. Configure data paths first.
Historical notebooks remain untouched in the two original folders.

In [ ]:
from pathlib import Path
import os
from inner_speech.config import load_config
from inner_speech.cli import run_stage
# Start Jupyter in the repository or notebooks directory.
root = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
config_path = Path(os.environ.get("INNER_SPEECH_CONFIG", root / "configs/default.yaml"))
cfg = load_config(config_path)


In [ ]:
run_stage(cfg, 'connectivity')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from inner_speech.connectivity import edge_matrix
p = Path(cfg['output_root']) / cfg['subjects'][0] / cfg['sessions'][0] / 'connectivity.npz'
with np.load(p, allow_pickle=False) as z:
    m = z['methods'].tolist().index('coh')
    n = len(z['channels'])
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for ax, key in zip(axes[:2], ['task', 'rest']):
        mat = edge_matrix(z[key][m, :, 0], z['edge_i'], z['edge_j'], n)
        im = ax.imshow(mat, vmin=0, vmax=1, cmap='viridis'); ax.set_title(key)
        fig.colorbar(im, ax=ax)
    delta = edge_matrix(z['task'][m,:,0]-z['rest'][m,:,0], z['edge_i'], z['edge_j'], n)
    im = axes[2].imshow(delta, vmin=-1, vmax=1, cmap='RdBu_r'); axes[2].set_title('Task minus rest (descriptive)')
    fig.colorbar(im, ax=axes[2]); fig.tight_layout()
